# EDA Star Wars Business Intelligence

Objetivo: explorar, limpiar y preparar dos datasets relacionados con Star Wars para construir un dashboard ejecutivo en Power BI.

## 1. Importacion de librerias


En esta sección se importan las librerías necesarias para trabajar con los datos.

También se configuran algunas opciones de visualización de pandas para poder ver más columnas y filas en el notebook.

Además, se crea la carpeta `data/clean/`, donde se guardarán los datasets limpios generados durante el proceso.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re
import unicodedata


def find_project_root(start_path=None):
    """Busca la raiz del proyecto para evitar rutas absolutas dependientes del usuario."""
    start_path = Path(start_path or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "data" / "raw").exists() and (candidate / "notebooks").exists():
            return candidate
    return start_path


BASE_DIR = find_project_root()
os.chdir(BASE_DIR)

print("Ahora Python esta en:")
print(Path.cwd())

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 1.2 Carga de datos

Coloca los CSV originales en `data/raw` y ajusta los nombres de archivo si hace falta.

In [ ]:
RAW_DIR = Path("data/raw") #definimos la carpeta base donde se encuentran los datos en bruto, 


# Creamos la ruta completa al CSV principal de la encuesta de Star Wars.
STARWARS_PATH = RAW_DIR / "StarWars" / "StarWars.csv" 


# Creamos la ruta a la carpeta donde están los CSV extra descargados de Kaggle.
KAGGLE_CSV_DIR = RAW_DIR / "starwars_kaggle" / "archive (1)" / "csv"

print("Ruta StarWars:", STARWARS_PATH) #mostramos por pantalla la ruta del archivo principal
print("Existe StarWars:", STARWARS_PATH.exists()) #devuelve True si el archivo existe, False si no

print("Ruta Kaggle CSV:", KAGGLE_CSV_DIR) 
print("Existe carpeta Kaggle CSV:", KAGGLE_CSV_DIR.exists()) #comprobamos si existe o no


#comprobamos si existe una archivo en especifico

print("Existe characters.csv:", (KAGGLE_CSV_DIR / "characters.csv").exists()) 

#pasamos a leer con pandas los archivos csv de las database que hemos elegido,
#el resultado se guarda en un dataframe para cada uno de los archivos
df_starwars_raw= pd.read_csv(STARWARS_PATH)

df_characters = pd.read_csv(KAGGLE_CSV_DIR / "characters.csv")
df_films = pd.read_csv(KAGGLE_CSV_DIR / "films.csv")
df_planets = pd.read_csv(KAGGLE_CSV_DIR / "planets.csv")
df_species = pd.read_csv(KAGGLE_CSV_DIR / "species.csv")
df_starships = pd.read_csv(KAGGLE_CSV_DIR / "starships.csv")
df_vehicles = pd.read_csv(KAGGLE_CSV_DIR / "vehicles.csv")
df_quotes = pd.read_csv(KAGGLE_CSV_DIR / "quotes.csv")
df_weapons = pd.read_csv(KAGGLE_CSV_DIR / "weapons.csv")
df_droids = pd.read_csv(KAGGLE_CSV_DIR / "droids.csv")


print("\nDatasets cargados correctamente:")

#mostramos los tamaños de los datasets cargados, el número de filas y columnas de cada uno

print("Characters:", df_characters.shape)
print("Films:", df_films.shape)
print("Planets:", df_planets.shape)
print("Species:", df_species.shape)
print("Starships:", df_starships.shape)
print("Vehicles:", df_vehicles.shape)
print("Quotes:", df_quotes.shape)
print("Weapons:", df_weapons.shape)
print("Droids:", df_droids.shape)
#mostramos el tamaño del dataset principal de la encuesta de star wars 
#y las primeras filas para comprobar que se ha cargado correctamente
print("Dimensiones originales:", df_starwars_raw.shape)
display(df_starwars_raw.head())


## x. Funciones generales de limpieza

Antes de limpiar cada dataset, se definen funciones reutilizables. Estas funciones permiten:

- Normalizar los nombres de las columnas a un formato común en inglés y `snake_case`.
- Aplicar diccionarios de equivalencias para que una misma variable tenga el mismo nombre en todos los datasets.
- Limpiar valores de texto vacíos o no informativos.
- Reutilizar la misma lógica en los datasets 


In [ ]:
# FUNCIONES GENERALES DE LIMPIEZA


def normalize_column_names(df):
    """Normaliza nombres de columnas a snake_case basico."""
    df = df.copy()
    normalized_columns = []

    for column in df.columns:
        column_name = str(column).strip().lower()
        column_name = unicodedata.normalize("NFKD", column_name).encode("ascii", "ignore").decode("ascii")
        column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
        column_name = re.sub(r"_+", "_", column_name).strip("_")
        normalized_columns.append(column_name)

    df.columns = normalized_columns
    return df


def clean_text(value):
    """Limpia un valor de texto individual y convierte falsos nulos en NaN."""
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    if value == "" or value.lower() in ["nan", "none", "null"]:
        return np.nan
    return value


def clean_text_columns(df):
    """Limpia todas las columnas de texto de un DataFrame."""
    df = df.copy()
    text_columns = df.select_dtypes(include=["object", "string"]).columns

    for column in text_columns:
        df[column] = df[column].apply(clean_text)

    return df


def apply_column_mapping(df, column_mapping):
    """Renombra columnas usando un diccionario de equivalencias."""
    df = df.copy()
    existing_mapping = {
        source_column: target_column
        for source_column, target_column in column_mapping.items()
        if source_column in df.columns
    }
    return df.rename(columns=existing_mapping)


def add_missing_columns(df, required_columns):
    """Anade columnas necesarias que no existan en el dataset."""
    df = df.copy()

    for column in required_columns:
        if column not in df.columns:
            df[column] = np.nan

    return df


def normalize_text_key(value):
    """Convierte cualquier etiqueta de texto en una clave estable tipo snake_case."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value if value else np.nan


CHARACTER_KEY_ALIASES = {
    "princess_leia_organa": "leia_organa",
    "c3po": "c_3po",
    "c_3_po": "c_3po",
    "r2d2": "r2_d2",
    "r2_d_2": "r2_d2",
    "obiwan_kenobi": "obi_wan_kenobi",
    "sheev_palpatine": "emperor_palpatine",
    "palpatine": "emperor_palpatine",
    "queen_amidala": "padme_amidala",
}

CHARACTER_DISPLAY_NAMES = {
    "han_solo": "Han Solo",
    "luke_skywalker": "Luke Skywalker",
    "leia_organa": "Leia Organa",
    "anakin_skywalker": "Anakin Skywalker",
    "obi_wan_kenobi": "Obi-Wan Kenobi",
    "emperor_palpatine": "Emperor Palpatine",
    "darth_vader": "Darth Vader",
    "lando_calrissian": "Lando Calrissian",
    "boba_fett": "Boba Fett",
    "c_3po": "C-3PO",
    "r2_d2": "R2-D2",
    "jar_jar_binks": "Jar Jar Binks",
    "padme_amidala": "Padme Amidala",
    "yoda": "Yoda",
}

FILM_KEY_ALIASES = {
    "episode_i_the_phantom_menace": "the_phantom_menace",
    "episode_ii_attack_of_the_clones": "attack_of_the_clones",
    "episode_iii_revenge_of_the_sith": "revenge_of_the_sith",
    "episode_iv_a_new_hope": "a_new_hope",
    "episode_v_the_empire_strikes_back": "the_empire_strikes_back",
    "episode_vi_return_of_the_jedi": "return_of_the_jedi",
    "episode_vii_the_force_awakens": "the_force_awakens",
    "episode_viii_the_last_jedi": "the_last_jedi",
    "episode_ix_the_rise_of_skywalker": "the_rise_of_skywalker",
    "star_wars_ep_i_the_phantom_menace": "the_phantom_menace",
    "star_wars_ep_ii_attack_of_the_clones": "attack_of_the_clones",
    "star_wars_ep_iii_revenge_of_the_sith": "revenge_of_the_sith",
    "star_wars_ep_iv_a_new_hope": "a_new_hope",
    "star_wars_ep_v_the_empire_strikes_back": "the_empire_strikes_back",
    "star_wars_ep_vi_return_of_the_jedi": "return_of_the_jedi",
    "star_wars_ep_vii_the_force_awakens": "the_force_awakens",
    "star_wars_ep_viii_the_last_jedi": "the_last_jedi",
    "star_wars_the_rise_of_skywalker": "the_rise_of_skywalker",
    "rogue_one_a_star_wars_story": "rogue_one",
    "solo_a_star_wars_story": "solo",
    "star_wars_the_mandalorian_and_grogu": "the_mandalorian_and_grogu",
}

FILM_DISPLAY_NAMES = {
    "the_phantom_menace": "Episode I: The Phantom Menace",
    "attack_of_the_clones": "Episode II: Attack of the Clones",
    "revenge_of_the_sith": "Episode III: Revenge of the Sith",
    "a_new_hope": "Episode IV: A New Hope",
    "the_empire_strikes_back": "Episode V: The Empire Strikes Back",
    "return_of_the_jedi": "Episode VI: Return of the Jedi",
    "the_force_awakens": "Episode VII: The Force Awakens",
    "the_last_jedi": "Episode VIII: The Last Jedi",
    "the_rise_of_skywalker": "Episode IX: The Rise of Skywalker",
    "rogue_one": "Rogue One: A Star Wars Story",
    "solo": "Solo: A Star Wars Story",
    "the_mandalorian_and_grogu": "The Mandalorian and Grogu",
}


def normalize_character_key(value):
    key = normalize_text_key(value)
    if pd.isna(key):
        return np.nan
    return CHARACTER_KEY_ALIASES.get(key, key)


def canonical_character_name(value):
    key = normalize_character_key(value)
    if pd.isna(key):
        return np.nan
    return CHARACTER_DISPLAY_NAMES.get(key, str(value).strip())


def normalize_film_key(value):
    key = normalize_text_key(value)
    if pd.isna(key):
        return np.nan
    return FILM_KEY_ALIASES.get(key, key)


def canonical_film_title(value):
    key = normalize_film_key(value)
    if pd.isna(key):
        return np.nan
    return FILM_DISPLAY_NAMES.get(key, str(value).strip())


def split_list_values(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(",") if item.strip()]


def normalize_list_text(value, canonical_func):
    items = split_list_values(value)
    if not items:
        return np.nan
    return ", ".join(canonical_func(item) for item in items)


def normalize_list_keys(value, key_func):
    items = split_list_values(value)
    if not items:
        return np.nan
    return ", ".join(key_func(item) for item in items)


def add_key_column(df, source_col, key_col, key_func=normalize_text_key):
    df = df.copy()
    if source_col in df.columns:
        df[key_col] = df[source_col].apply(key_func)
    return df


# 2. Estudio y analisis datasec/encuesta Starwars


Antes de limpiar o analizar el dataset de la encuesta de Star Wars, realizamos una revisión estructural porque detectamos que no se comportaba como un CSV tradicional.

Al cargar el archivo observamos que muchas columnas aparecían con nombres automáticos como `unnamed_4`, `unnamed_5`, `unnamed_18`, etc. Esto no significaba necesariamente que fueran columnas inútiles o vacías, sino que formaban parte de preguntas agrupadas. Es decir, algunas preguntas de la encuesta ocupaban varias columnas, una por cada opción de respuesta.

También detectamos que la fila 0 del dataset no era una respuesta real de una persona encuestada. Esa primera fila contenía información auxiliar o metadata sobre las opciones de algunas preguntas. Por ejemplo, en las columnas de películas vistas y ranking, la fila 0 indicaba a qué episodio correspondía cada columna. En las columnas de opinión sobre personajes, la fila 0 permitía identificar personajes como Anakin Skywalker, Obi Wan Kenobi, Darth Vader, Yoda, etc.

Por este motivo, tomamos la decisión de separar el dataset en dos partes:

- `df_starwars_raw`: copia original del dataset sin modificar.
- `metadata_row`: primera fila del dataset, guardada como referencia para interpretar columnas.
- `df_survey`: versión de trabajo de la encuesta, eliminando la fila 0 para quedarnos solo con respuestas reales.

Después creamos un diccionario o mapa de columnas (`column_dictionary`) para comparar la posición de cada columna, su nombre original, el valor de la fila 0 y el porcentaje de valores nulos. Este paso nos permitió entender qué representaba cada columna antes de aplicar transformaciones.

### Conclusiones del análisis estructural

Tras revisar columnas, valores únicos y la fila 0, concluimos que la encuesta está organizada en varios bloques:

1. **Preguntas generales iniciales**  
   Incluyen si la persona ha visto alguna película de Star Wars y si se considera fan de la franquicia.

2. **Películas vistas**  
   Corresponden a las columnas 3 a 8. Aunque algunas aparecen como `unnamed`, la fila 0 indica qué película representa cada columna. En estas columnas, si aparece el nombre de la película, significa que la persona la ha visto; si aparece un valor nulo, significa que no la ha seleccionado.

3. **Ranking de películas**  
   Corresponde a las columnas 9 a 14. Estas columnas contienen valores del 1 al 6, donde 1 representa la película favorita y 6 la menos favorita. Por tanto, no deben tratarse como texto normal, sino como variables numéricas de ranking.

4. **Opinión sobre personajes**  
   Corresponde a las columnas 15 a 28. Estas columnas contienen respuestas como `very favorably`, `somewhat favorably`, `neutral`, `somewhat unfavorably`, `very unfavorably` o `unfamiliar`. Los nombres reales de los personajes se obtienen a partir de la fila 0.

5. **Preguntas finales y datos demográficos**  
   A partir de la columna 29 detectamos un desplazamiento entre el nombre de la columna y el contenido real. Por ejemplo, la columna `unnamed_29` contenía respuestas como `han`, `greedo` o `i don't understand this question`, por lo que decidimos interpretarla como la pregunta `which_character_shot_first`.

### Decisiones tomadas

Debido a esta estructura especial, decidimos no aplicar una limpieza automática directamente sobre todo el dataset original. En su lugar, seguimos este proceso:

1. Conservar una copia original del dataset.
2. Guardar la fila 0 como metadata.
3. Eliminar la fila 0 de la versión de trabajo.
4. Crear un mapa de columnas para interpretar las columnas `unnamed`.
5. Agrupar las columnas por bloques temáticos.
6. Renombrar manualmente las columnas importantes usando la información de la fila 0 y los valores únicos observados.
7. Analizar cada bloque según su naturaleza:
   - preguntas sí/no como variables categóricas,
   - películas vistas como selección múltiple,
   - ranking de películas como valores numéricos,
   - opiniones de personajes como variables ordinales/categóricas,
   - datos demográficos revisados con especial cuidado por el desplazamiento detectado.

Esta revisión previa es necesaria para evitar errores de interpretación. Si hubiéramos tratado el dataset como una tabla normal desde el principio, podríamos haber analizado columnas `unnamed` sin saber qué representaban realmente o haber usado nombres de columnas que no coincidían con el contenido real.

In [ ]:

# 1. INSPECCIÓN INICIAL DE LA ESTRUCTURA


print("Columnas del dataset original:\n")

for i, col in enumerate(df_starwars_raw.columns):
    print(i, "->", col)



In [ ]:

# INSPECCIÓN ESPECIAL DEL DATASET DE ENCUESTA STAR WARS


# Guardamos una copia del dataset original de la encuesta.
# Así no perdemos nunca la estructura tal como venía en el CSV.

df_survey = df_starwars_raw.copy()

# Guardamos la primera fila porque parece contener información de opciones/subpreguntas.
# No parece una respuesta real de una persona.

metadata_row = df_starwars_raw.iloc[0]

# Creamos una versión de trabajo de la encuesta quitando la fila 0.
# Esta será la tabla con respuestas reales.

df_survey = df_starwars_raw.drop(index=0).reset_index(drop=True)
print("Dimensiones originales:", df_starwars_raw.shape)
print("Dimensiones encuesta limpia de filas:", df_survey.shape)

display(df_survey.head())



In [ ]:

# REPORTE DE NULOS DE LA ENCUESTA STAR WARS


def missing_report(df):
    """
    Crea un informe de valores nulos por columna.
    Muestra:
    - número de nulos
    - porcentaje de nulos
    Solo muestra columnas que tienen al menos un nulo.
    """
    return (
        pd.DataFrame({
            "nulos": df.isna().sum(),
            "porcentaje": (df.isna().mean() * 100).round(2),
        })
        .query("nulos > 0")
        .sort_values("porcentaje", ascending=False)
    )


# Aplicamos el reporte solo a la encuesta limpia, sin la fila 0 de metadata.
missing_survey = missing_report(df_survey)

display(missing_survey)

In [ ]:

# 4. MAPA DE COLUMNAS DE LA ENCUESTA

column_dictionary = pd.DataFrame({
    "posicion": range(len(df_starwars_raw.columns)),
    "columna_original": df_starwars_raw.columns,
    "metadata_fila_0": metadata_row.values,
    "nulos_en_respuestas": df_survey.isna().sum().values,
    "porcentaje_nulos_en_respuestas": (df_survey.isna().mean() * 100).round(2).values
})

display(column_dictionary)

In [ ]:

# 5. REVISIÓN DE COLUMNAS UNNAMED


unnamed_columns = [
    col for col in df_starwars_raw.columns 
    if "unnamed" in str(col).lower()
]

print("Número de columnas Unnamed:", len(unnamed_columns))

unnamed_dictionary = column_dictionary[
    column_dictionary["columna_original"].isin(unnamed_columns)
]

display(unnamed_dictionary)

In [ ]:

# 6. NORMALIZAR NOMBRES DE COLUMNAS DE LA ENCUESTA


df_survey = normalize_column_names(df_survey)

print("Columnas normalizadas:")
for i, col in enumerate(df_survey.columns):
    print(i, "->", col)

In [ ]:

# 7. LIMPIEZA BÁSICA DE TEXTOS
'''quita espacios al principio/final
convierte textos vacíos en NaN
convierte "nan", "none", "null" en NaN real
'''

df_survey = clean_text_columns(df_survey)

display(df_survey.head())

In [ ]:

# 9. REVISIÓN DE VALORES ÚNICOS POR COLUMNA

'''nombre de columna
número de valores distintos
primeros valores únicos'''

for col in df_survey.columns:
    print("\n" + "=" * 80)
    print("COLUMNA:", col)
    print("Nº valores únicos:", df_survey[col].nunique(dropna=True))
    print(df_survey[col].dropna().unique()[:10])

In [ ]:

# 10. IDENTIFICAR BLOQUES DE COLUMNAS POR POSICIÓN

#Con esta salida vamos a decidir qué columnas pertenecen a cada bloque

for i, col in enumerate(df_survey.columns):
    print(i, "->", col)

### Revisión de columnas desplazadas en la encuesta

Después de realizar el análisis estructural anterior, detectamos que a partir de la columna 29 existe un desplazamiento en los encabezados de la encuesta. Los nombres de las columnas no coinciden con el contenido real que aparece en sus valores.

Concretamente, observamos que las respuestas de la columna 29 corresponden realmente a la pregunta que aparece nombrada en la columna 31. Es decir, desde ese punto los encabezados parecen estar desplazados dos posiciones hacia la derecha:

- La pregunta de la columna 31 corresponde realmente a los valores de la columna 29.
- La pregunta de la columna 32 corresponde realmente a los valores de la columna 30.
- La pregunta de la columna 33 corresponde realmente a los valores de la columna 31.
- Y así sucesivamente en las columnas finales del dataset.

Por este motivo, decidimos no fiarnos únicamente del nombre original de las columnas, sino analizar también sus valores únicos para identificar qué representa realmente cada una.

Un ejemplo claro es la columna 29, que aparece como `unnamed_29`, pero contiene respuestas como `han`, `greedo` o `i don't understand this question`. Por tanto, esta columna se interpreta como la pregunta `which_character_shot_first`.

El desplazamiento afecta especialmente a las preguntas finales y a las variables demográficas. Sin embargo, en las dos últimas columnas del dataset encontramos una dificultad adicional: ambas contienen respuestas relacionadas con lugares geográficos o regiones, por lo que no queda completamente claro a qué variable original corresponde cada una.

### Decisión sobre las dos últimas columnas

Como las dos últimas columnas contienen valores geográficos, se decide tratarlas con precaución. En lugar de eliminarlas directamente, se conservarán temporalmente para analizarlas con más detalle.

La decisión inicial será:

1. Revisar sus valores únicos.
2. Compararlas con las categorías esperadas de `location_census_region`.
3. Comprobar si una de ellas contiene claramente regiones censales válidas.
4. Mantener como `location_census_region` la columna que tenga mayor coherencia geográfica.
5. Dejar la otra como columna auxiliar o descartarla si se confirma que es duplicada, residual o resultado del desplazamiento de encabezados.

Por tanto, estas columnas no se eliminarán en la primera limpieza. Se mantendrán como columnas pendientes de validación hasta confirmar si aportan información útil o si deben excluirse del análisis final.

In [ ]:

# AGRUPACIÓN DE COLUMNAS DE LA ENCUESTA STAR WARS

# Preguntas generales principales.
# Incluyen preguntas de sí/no y preguntas generales de cultura Star Wars.
general_columns = [
    df_survey.columns[1],   # ha visto alguna película de Star Wars
    df_survey.columns[2],   # se considera fan de Star Wars
    df_survey.columns[29],  # quién disparó primero
    df_survey.columns[30],  # familiaridad con universo expandido
    df_survey.columns[31],  # fan del universo expandido
    df_survey.columns[32],  # fan de Star Trek
]

# Columnas de películas vistas.
# Estas columnas indican qué películas ha visto cada persona.
# Si hay valor, normalmente significa que la ha visto; si hay NaN, no la ha visto.
seen_movies_columns = list(df_survey.columns[3:9])

# Columnas de ranking de películas.
# Valores del 1 al 6, donde 1 suele significar favorita y 6 menos favorita.
ranking_columns = list(df_survey.columns[9:15])

# Columnas de opinión sobre personajes.
# Contienen valores tipo:
# very favorably, somewhat favorably, neutral, unfavorable, unfamiliar...
character_opinion_columns = list(df_survey.columns[15:29])

# Columnas demográficas.
# OJO: en este dataset algunas columnas vienen desplazadas,
# por eso después las renombraremos manualmente.
demographic_columns = list(df_survey.columns[33:40])


# Guardamos todos los grupos en un diccionario para poder revisarlos fácilmente.
survey_column_groups = {
    "general": general_columns,
    "seen_movies": seen_movies_columns,
    "ranking": ranking_columns,
    "character_opinion": character_opinion_columns,
    "demographic": demographic_columns,
}


# Mostramos los grupos para comprobar que están bien.
for group_name, columns in survey_column_groups.items():
    print("\n" + "=" * 70)
    print(group_name.upper())
    print("=" * 70)
    for col in columns:
        print(col)


In [ ]:

# RENOMBRADO MANUAL DE COLUMNAS DE LA ENCUESTA


survey_column_mapping = {
    # Identificador
    df_survey.columns[0]: "respondent_id",

    # Preguntas generales
    df_survey.columns[1]: "has_seen_any_star_wars_film",
    df_survey.columns[2]: "is_star_wars_fan",

    # Películas vistas
    df_survey.columns[3]: "seen_episode_i_the_phantom_menace",
    df_survey.columns[4]: "seen_episode_ii_attack_of_the_clones",
    df_survey.columns[5]: "seen_episode_iii_revenge_of_the_sith",
    df_survey.columns[6]: "seen_episode_iv_a_new_hope",
    df_survey.columns[7]: "seen_episode_v_the_empire_strikes_back",
    df_survey.columns[8]: "seen_episode_vi_return_of_the_jedi",

    # Ranking de películas
    df_survey.columns[9]: "rank_episode_i_the_phantom_menace",
    df_survey.columns[10]: "rank_episode_ii_attack_of_the_clones",
    df_survey.columns[11]: "rank_episode_iii_revenge_of_the_sith",
    df_survey.columns[12]: "rank_episode_iv_a_new_hope",
    df_survey.columns[13]: "rank_episode_v_the_empire_strikes_back",
    df_survey.columns[14]: "rank_episode_vi_return_of_the_jedi",

    # Opinión sobre personajes
    df_survey.columns[15]: "opinion_han_solo",
    df_survey.columns[16]: "opinion_luke_skywalker",
    df_survey.columns[17]: "opinion_princess_leia_organa",
    df_survey.columns[18]: "opinion_anakin_skywalker",
    df_survey.columns[19]: "opinion_obi_wan_kenobi",
    df_survey.columns[20]: "opinion_emperor_palpatine",
    df_survey.columns[21]: "opinion_darth_vader",
    df_survey.columns[22]: "opinion_lando_calrissian",
    df_survey.columns[23]: "opinion_boba_fett",
    df_survey.columns[24]: "opinion_c_3po",
    df_survey.columns[25]: "opinion_r2_d2",
    df_survey.columns[26]: "opinion_jar_jar_binks",
    df_survey.columns[27]: "opinion_padme_amidala",
    df_survey.columns[28]: "opinion_yoda",

    # Preguntas generales finales
    df_survey.columns[29]: "which_character_shot_first",
    df_survey.columns[30]: "is_familiar_with_expanded_universe",
    df_survey.columns[31]: "is_expanded_universe_fan",
    df_survey.columns[32]: "is_star_trek_fan",

    # Datos demográficos
    df_survey.columns[33]: "gender",
    df_survey.columns[34]: "age",
    df_survey.columns[35]: "household_income",
    df_survey.columns[36]: "education",
    df_survey.columns[37]: "location_census_region",
    df_survey.columns[38]: "extra_column_38",
    df_survey.columns[39]: "extra_column_39",
}

df_survey = df_survey.rename(columns=survey_column_mapping)

print("Columnas renombradas correctamente:")
for i, col in enumerate(df_survey.columns):
    print(i, "->", col)



In [ ]:

# GRUPOS DE COLUMNAS CON NOMBRES DEFINITIVOS


general_columns = [
    "has_seen_any_star_wars_film",
    "is_star_wars_fan",
    "which_character_shot_first",
    "is_familiar_with_expanded_universe",
    "is_expanded_universe_fan",
    "is_star_trek_fan",
]

seen_movies_columns = [
    "seen_episode_i_the_phantom_menace",
    "seen_episode_ii_attack_of_the_clones",
    "seen_episode_iii_revenge_of_the_sith",
    "seen_episode_iv_a_new_hope",
    "seen_episode_v_the_empire_strikes_back",
    "seen_episode_vi_return_of_the_jedi",
]

ranking_columns = [
    "rank_episode_i_the_phantom_menace",
    "rank_episode_ii_attack_of_the_clones",
    "rank_episode_iii_revenge_of_the_sith",
    "rank_episode_iv_a_new_hope",
    "rank_episode_v_the_empire_strikes_back",
    "rank_episode_vi_return_of_the_jedi",
]

character_opinion_columns = [
    "opinion_han_solo",
    "opinion_luke_skywalker",
    "opinion_princess_leia_organa",
    "opinion_anakin_skywalker",
    "opinion_obi_wan_kenobi",
    "opinion_emperor_palpatine",
    "opinion_darth_vader",
    "opinion_lando_calrissian",
    "opinion_boba_fett",
    "opinion_c_3po",
    "opinion_r2_d2",
    "opinion_jar_jar_binks",
    "opinion_padme_amidala",
    "opinion_yoda",
]

demographic_columns = [
    "gender",
    "age",
    "household_income",
    "education",
    "location_census_region",
]

survey_column_groups = {
    "general": general_columns,
    "seen_movies": seen_movies_columns,
    "ranking": ranking_columns,
    "character_opinion": character_opinion_columns,
    "demographic": demographic_columns,
}

for group_name, columns in survey_column_groups.items():
    print("\n" + "=" * 70)
    print(group_name.upper())
    print("=" * 70)
    for col in columns:
        print(col)

In [ ]:

# REVISIÓN DE VALORES ÚNICOS TRAS RENOMBRAR


for group_name, columns in survey_column_groups.items():
    print("\n" + "=" * 80)
    print(f"GRUPO: {group_name.upper()}")
    print("=" * 80)

    for col in columns:
        if col in df_survey.columns:
            print("\nCOLUMNA:", col)
            print("Nº valores únicos:", df_survey[col].nunique(dropna=True))
            print(df_survey[col].dropna().unique()[:10])

## 3. Limpieza final de la encuesta

Este bloque convierte el diagnostico estructural en una limpieza reproducible para Power BI: demografia corregida, peliculas vistas en 1/0, rankings numericos, opiniones puntuadas y variables auxiliares de negocio.

In [ ]:
# LIMPIEZA FINAL DE LA ENCUESTA

# Partimos de df_survey, que ya tiene la fila 0 eliminada y las columnas renombradas.
df_survey_clean = df_survey.copy()

# 1. Reconstruccion de demografia.
# Los ingresos venian partidos por comas: $50,000 - $99,999 se reparte en varias columnas.
VALID_GENDERS = {"male", "female"}
VALID_AGES = {"18-29", "30-44", "45-60", "> 60"}
VALID_EDUCATION = {
    "less than high school degree",
    "high school degree",
    "some college or associate degree",
    "bachelor degree",
    "graduate degree",
}
VALID_REGIONS = {
    "new england", "middle atlantic", "east north central", "west north central",
    "south atlantic", "east south central", "west south central", "mountain", "pacific",
}


def _clean_token(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    return np.nan if value in ["", "nan", "none", "null"] else value


def _first_valid(tokens, valid_values):
    for token in tokens:
        if token in valid_values:
            return token
    return np.nan


def _rebuild_income(tokens):
    if "$0 - $24" in tokens and "999" in tokens:
        return "$0 - $24,999"
    if "$25" in tokens and "000 - $49" in tokens and "999" in tokens:
        return "$25,000 - $49,999"
    if "$50" in tokens and "000 - $99" in tokens and "999" in tokens:
        return "$50,000 - $99,999"
    if "$100" in tokens and "000 - $149" in tokens and "999" in tokens:
        return "$100,000 - $149,999"
    if "$150" in tokens and "000+" in tokens:
        return "$150,000+"
    return np.nan


def _rebuild_demographics(row):
    tail_cols = [
        "is_expanded_universe_fan", "is_star_trek_fan", "gender", "age",
        "household_income", "education", "location_census_region",
        "extra_column_38", "extra_column_39",
    ]
    tokens = [_clean_token(row[col]) for col in tail_cols if col in row.index]
    tokens = [token for token in tokens if isinstance(token, str)]
    return pd.Series({
        "gender": _first_valid(tokens, VALID_GENDERS),
        "age": _first_valid(tokens, VALID_AGES),
        "household_income": _rebuild_income(tokens),
        "education": _first_valid(tokens, VALID_EDUCATION),
        "location_census_region": _first_valid(tokens, VALID_REGIONS),
    })

fixed_demographics = df_survey_clean.apply(_rebuild_demographics, axis=1)
for col in fixed_demographics.columns:
    df_survey_clean[col] = fixed_demographics[col]

df_survey_clean = df_survey_clean.drop(
    columns=[col for col in ["extra_column_38", "extra_column_39"] if col in df_survey_clean.columns]
)

# 2. Peliculas vistas: texto = vista, NaN = no seleccionada.
for col in seen_movies_columns:
    df_survey_clean[col] = df_survey_clean[col].notna().astype(int)

df_survey_clean["total_movies_seen"] = df_survey_clean[seen_movies_columns].sum(axis=1)

# 3. Rankings de peliculas a numerico.
for col in ranking_columns:
    df_survey_clean[col] = pd.to_numeric(df_survey_clean[col], errors="coerce")

df_survey_clean["has_complete_movie_ranking"] = (
    df_survey_clean[ranking_columns].notna().sum(axis=1).eq(6).astype(int)
)

# 4. Opiniones de personajes a puntuacion numerica.
opinion_score_map = {
    "very favorably": 2,
    "somewhat favorably": 1,
    "neither favorably nor unfavorably (neutral)": 0,
    "somewhat unfavorably": -1,
    "very unfavorably": -2,
    "unfamiliar (n/a)": np.nan,
}
opinion_score_columns = []
for col in character_opinion_columns:
    score_col = col.replace("opinion_", "opinion_score_")
    df_survey_clean[score_col] = df_survey_clean[col].map(opinion_score_map)
    opinion_score_columns.append(score_col)

df_survey_clean["average_character_opinion_score"] = df_survey_clean[opinion_score_columns].mean(axis=1)

# 5. Variables auxiliares para analisis ejecutivo.
def yes_no_to_binary(value):
    if value == "yes":
        return 1
    if value == "no":
        return 0
    return np.nan

for col in [
    "has_seen_any_star_wars_film", "is_star_wars_fan",
    "is_familiar_with_expanded_universe", "is_expanded_universe_fan", "is_star_trek_fan",
]:
    df_survey_clean[col + "_binary"] = df_survey_clean[col].apply(yes_no_to_binary)


def movie_consumption_segment(total_movies):
    if total_movies == 0:
        return "no_movies_seen"
    if total_movies <= 2:
        return "low_consumption"
    if total_movies <= 4:
        return "medium_consumption"
    return "high_consumption"


df_survey_clean["movie_consumption_segment"] = df_survey_clean["total_movies_seen"].apply(movie_consumption_segment)
df_survey_clean["fan_segment"] = np.where(
    df_survey_clean["is_star_wars_fan"] == "yes", "fan",
    np.where(df_survey_clean["is_star_wars_fan"] == "no", "not_fan", "unknown")
)
df_survey_clean["has_demographic_info"] = df_survey_clean[[
    "gender", "age", "household_income", "education", "location_census_region"
]].notna().any(axis=1).astype(int)

print("Dimensiones encuesta limpia:", df_survey_clean.shape)
print("Duplicados completos:", df_survey_clean.duplicated().sum())
print("IDs duplicados:", df_survey_clean["respondent_id"].duplicated().sum())

print("\nDistribucion de peliculas vistas:")
display(df_survey_clean["total_movies_seen"].value_counts().sort_index())

print("\nCategorias demograficas corregidas:")
for col in ["gender", "age", "household_income", "education", "location_census_region"]:
    print("\n" + col)
    display(df_survey_clean[col].value_counts(dropna=False))

print("\nNulos principales tras limpieza:")
display(missing_report(df_survey_clean).head(20))

## 4. Tablas largas para Power BI

Para facilitar filtros, rankings y relaciones en Power BI, la encuesta se separa en una tabla de respondentes y tres tablas largas: peliculas vistas, ranking de peliculas y opiniones de personajes.

In [ ]:
# TABLAS LARGAS PARA POWER BI

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

movie_metadata = {
    "seen_episode_i_the_phantom_menace": ("Episode I: The Phantom Menace", "the_phantom_menace", 1),
    "seen_episode_ii_attack_of_the_clones": ("Episode II: Attack of the Clones", "attack_of_the_clones", 2),
    "seen_episode_iii_revenge_of_the_sith": ("Episode III: Revenge of the Sith", "revenge_of_the_sith", 3),
    "seen_episode_iv_a_new_hope": ("Episode IV: A New Hope", "a_new_hope", 4),
    "seen_episode_v_the_empire_strikes_back": ("Episode V: The Empire Strikes Back", "the_empire_strikes_back", 5),
    "seen_episode_vi_return_of_the_jedi": ("Episode VI: Return of the Jedi", "return_of_the_jedi", 6),
}
rank_metadata = {col: movie_metadata[col.replace("rank_", "seen_")] for col in ranking_columns}

character_metadata = {
    "opinion_han_solo": ("han_solo", "Han Solo"),
    "opinion_luke_skywalker": ("luke_skywalker", "Luke Skywalker"),
    "opinion_princess_leia_organa": ("leia_organa", "Leia Organa"),
    "opinion_anakin_skywalker": ("anakin_skywalker", "Anakin Skywalker"),
    "opinion_obi_wan_kenobi": ("obi_wan_kenobi", "Obi-Wan Kenobi"),
    "opinion_emperor_palpatine": ("emperor_palpatine", "Emperor Palpatine"),
    "opinion_darth_vader": ("darth_vader", "Darth Vader"),
    "opinion_lando_calrissian": ("lando_calrissian", "Lando Calrissian"),
    "opinion_boba_fett": ("boba_fett", "Boba Fett"),
    "opinion_c_3po": ("c_3po", "C-3PO"),
    "opinion_r2_d2": ("r2_d2", "R2-D2"),
    "opinion_jar_jar_binks": ("jar_jar_binks", "Jar Jar Binks"),
    "opinion_padme_amidala": ("padme_amidala", "Padme Amidala"),
    "opinion_yoda": ("yoda", "Yoda"),
}

survey_respondents = df_survey_clean[[
    "respondent_id", "has_seen_any_star_wars_film", "has_seen_any_star_wars_film_binary",
    "is_star_wars_fan", "is_star_wars_fan_binary", "fan_segment",
    "total_movies_seen", "movie_consumption_segment", "has_complete_movie_ranking",
    "which_character_shot_first", "is_familiar_with_expanded_universe",
    "is_familiar_with_expanded_universe_binary", "is_expanded_universe_fan",
    "is_expanded_universe_fan_binary", "is_star_trek_fan", "is_star_trek_fan_binary",
    "gender", "age", "household_income", "education", "location_census_region",
    "has_demographic_info", "average_character_opinion_score",
]].copy()

survey_movies_seen = df_survey_clean[["respondent_id"] + seen_movies_columns].melt(
    id_vars="respondent_id", var_name="movie_code", value_name="has_seen_movie"
)
survey_movies_seen["movie_title"] = survey_movies_seen["movie_code"].map({k: v[0] for k, v in movie_metadata.items()})
survey_movies_seen["film_key"] = survey_movies_seen["movie_code"].map({k: v[1] for k, v in movie_metadata.items()})
survey_movies_seen["episode_order"] = survey_movies_seen["movie_code"].map({k: v[2] for k, v in movie_metadata.items()})

survey_movie_rankings = df_survey_clean[["respondent_id"] + ranking_columns].melt(
    id_vars="respondent_id", var_name="movie_code", value_name="movie_rank"
)
survey_movie_rankings["movie_title"] = survey_movie_rankings["movie_code"].map({k: v[0] for k, v in rank_metadata.items()})
survey_movie_rankings["film_key"] = survey_movie_rankings["movie_code"].map({k: v[1] for k, v in rank_metadata.items()})
survey_movie_rankings["episode_order"] = survey_movie_rankings["movie_code"].map({k: v[2] for k, v in rank_metadata.items()})
survey_movie_rankings["has_ranking"] = survey_movie_rankings["movie_rank"].notna().astype(int)

survey_character_opinions = df_survey_clean[["respondent_id"] + character_opinion_columns].melt(
    id_vars="respondent_id", var_name="character_code", value_name="opinion_label"
)
survey_character_opinions["character_key"] = survey_character_opinions["character_code"].map({k: v[0] for k, v in character_metadata.items()})
survey_character_opinions["character_name"] = survey_character_opinions["character_code"].map({k: v[1] for k, v in character_metadata.items()})
survey_character_opinions["opinion_score"] = survey_character_opinions["opinion_label"].map(opinion_score_map)
survey_character_opinions["has_character_opinion"] = survey_character_opinions["opinion_label"].notna().astype(int)
survey_character_opinions["is_unfamiliar"] = survey_character_opinions["opinion_label"].eq("unfamiliar (n/a)").astype(int)

print("survey_respondents:", survey_respondents.shape)
print("survey_movies_seen:", survey_movies_seen.shape)
print("survey_movie_rankings:", survey_movie_rankings.shape)
print("survey_character_opinions:", survey_character_opinions.shape)

display(survey_respondents.head())
display(survey_movies_seen.head())
display(survey_movie_rankings.head())
display(survey_character_opinions.head())

# Exportacion provisional para Power BI.
df_survey_clean.to_csv(PROCESSED_DIR / "survey_clean_wide.csv", index=False)
survey_respondents.to_csv(PROCESSED_DIR / "survey_respondents.csv", index=False)
survey_movies_seen.to_csv(PROCESSED_DIR / "survey_movies_seen.csv", index=False)
survey_movie_rankings.to_csv(PROCESSED_DIR / "survey_movie_rankings.csv", index=False)
survey_character_opinions.to_csv(PROCESSED_DIR / "survey_character_opinions.csv", index=False)

print("CSV de encuesta exportados en:", PROCESSED_DIR)


## 5. Revision general inicial de datasets del universo


In [ ]:
df_characters.info()
df_characters.head()


In [ ]:
df_planets.info()
df_planets.head()

In [ ]:
df_species.info()
df_species.head()


In [ ]:
df_starships.info()
df_starships.head()

In [ ]:
df_vehicles.info()
df_vehicles.head()

In [ ]:
df_quotes.info()
df_quotes.head()

In [ ]:
df_weapons.info()
df_weapons.head()

In [ ]:
df_droids.info()
df_droids.head()

## 6. Limpieza y preparacion de datasets del universo

En esta fase se limpian los datasets internos del universo Star Wars para usarlos en Power BI.

La limpieza aplicada es comun y reproducible:

- Normalizacion de nombres de columnas.
- Limpieza de textos y falsos nulos (`unknown`, `none`, `n/a`, cadenas vacias).
- Conversion de columnas numericas.
- Conversion de fechas de peliculas.
- Revision de duplicados y nulos.
- Creacion de columnas auxiliares de presencia: numero de peliculas asociadas, numero de residentes, pilotos o apariciones cuando aplica.
- Exportacion de CSV limpios a `data/processed`.

In [ ]:
# LIMPIEZA GENERAL DE DATASETS DEL UNIVERSO

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

UNKNOWN_VALUES = {
    "", "unknown", "n/a", "na", "none", "null", "nan", "not available", "not applicable"
}


def clean_universe_text_columns(df):
    """Limpia textos y convierte falsos nulos en NaN."""
    df = df.copy()
    text_columns = df.select_dtypes(include=["object", "string", "str"]).columns

    for col in text_columns:
        df[col] = df[col].apply(clean_text)
        df[col] = df[col].apply(
            lambda value: np.nan
            if isinstance(value, str) and value.strip().lower() in UNKNOWN_VALUES
            else value
        )

    return df


def to_numeric_columns(df, columns):
    """Convierte columnas a numerico si existen."""
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def count_list_items(value):
    """Cuenta elementos separados por coma en columnas tipo lista."""
    if pd.isna(value):
        return 0
    value = str(value).strip()
    if value == "":
        return 0
    if value.lower() == "all episodes":
        return 11
    return len([item for item in value.split(",") if item.strip()])


def add_list_count(df, source_col, target_col):
    """Crea una columna con numero de elementos en una lista textual."""
    df = df.copy()
    if source_col in df.columns:
        df[target_col] = df[source_col].apply(count_list_items)
    return df


def clean_year_column(value):
    """Extrae valores numericos de anos tipo '0 BBY' o '34'."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if not match:
        return np.nan
    return float(match.group())


universe_raw_datasets = {
    "characters": df_characters,
    "films": df_films,
    "planets": df_planets,
    "species": df_species,
    "starships": df_starships,
    "vehicles": df_vehicles,
    "quotes": df_quotes,
    "weapons": df_weapons,
    "droids": df_droids,
}

universe_clean_datasets = {}

for dataset_name, dataset in universe_raw_datasets.items():
    df = normalize_column_names(dataset)
    df = clean_universe_text_columns(df)
    df = df.drop_duplicates().reset_index(drop=True)
    universe_clean_datasets[dataset_name] = df

# Conversiones especificas por dataset.

universe_clean_datasets["characters"] = to_numeric_columns(
    universe_clean_datasets["characters"],
    ["id", "height", "weight"],
)
for col in ["year_born", "year_died"]:
    if col in universe_clean_datasets["characters"].columns:
        universe_clean_datasets["characters"][col] = universe_clean_datasets["characters"][col].apply(clean_year_column)

universe_clean_datasets["films"]["release_date"] = pd.to_datetime(
    universe_clean_datasets["films"]["release_date"], errors="coerce"
)
universe_clean_datasets["films"]["release_year"] = universe_clean_datasets["films"]["release_date"].dt.year

universe_clean_datasets["planets"] = to_numeric_columns(
    universe_clean_datasets["planets"],
    ["id", "diameter", "rotation_period", "orbital_period", "population", "surface_water"],
)
universe_clean_datasets["planets"] = add_list_count(universe_clean_datasets["planets"], "residents", "resident_count")
universe_clean_datasets["planets"] = add_list_count(universe_clean_datasets["planets"], "films", "film_count")

universe_clean_datasets["species"] = to_numeric_columns(
    universe_clean_datasets["species"],
    ["id", "average_height", "average_lifespan"],
)

universe_clean_datasets["starships"] = to_numeric_columns(
    universe_clean_datasets["starships"],
    [
        "id", "cost_in_credits", "length", "max_atmosphering_speed", "crew",
        "passengers", "cargo_capacity", "hyperdrive_rating", "mglt", "MGLT",
    ],
)
universe_clean_datasets["starships"] = add_list_count(universe_clean_datasets["starships"], "pilots", "pilot_count")
universe_clean_datasets["starships"] = add_list_count(universe_clean_datasets["starships"], "films", "film_count")

universe_clean_datasets["vehicles"] = to_numeric_columns(
    universe_clean_datasets["vehicles"],
    ["id", "cost_in_credits", "length", "max_atmosphering_speed", "crew", "passengers", "cargo_capacity"],
)
universe_clean_datasets["vehicles"] = add_list_count(universe_clean_datasets["vehicles"], "pilots", "pilot_count")
universe_clean_datasets["vehicles"] = add_list_count(universe_clean_datasets["vehicles"], "films", "film_count")

universe_clean_datasets["quotes"] = to_numeric_columns(universe_clean_datasets["quotes"], ["id"])

universe_clean_datasets["weapons"] = to_numeric_columns(
    universe_clean_datasets["weapons"],
    ["id", "cost_in_credits", "length"],
)
universe_clean_datasets["weapons"] = add_list_count(universe_clean_datasets["weapons"], "films", "film_count")

universe_clean_datasets["droids"] = to_numeric_columns(
    universe_clean_datasets["droids"],
    ["id", "height", "mass"],
)
universe_clean_datasets["droids"] = add_list_count(universe_clean_datasets["droids"], "films", "film_count")

# Claves normalizadas para relaciones y etiquetas canonicas.
universe_clean_datasets["characters"]["character_key"] = universe_clean_datasets["characters"]["name"].apply(normalize_character_key)
universe_clean_datasets["characters"]["name"] = universe_clean_datasets["characters"]["name"].apply(canonical_character_name)

universe_clean_datasets["films"]["film_key"] = universe_clean_datasets["films"]["title"].apply(normalize_film_key)
universe_clean_datasets["films"]["title"] = universe_clean_datasets["films"]["title"].apply(canonical_film_title)

universe_clean_datasets["planets"] = add_key_column(universe_clean_datasets["planets"], "name", "planet_key")
universe_clean_datasets["planets"]["resident_keys"] = universe_clean_datasets["planets"]["residents"].apply(lambda value: normalize_list_keys(value, normalize_character_key))
universe_clean_datasets["planets"]["residents"] = universe_clean_datasets["planets"]["residents"].apply(lambda value: normalize_list_text(value, canonical_character_name))
universe_clean_datasets["planets"]["film_keys"] = universe_clean_datasets["planets"]["films"].apply(lambda value: normalize_list_keys(value, normalize_film_key))

universe_clean_datasets["species"] = add_key_column(universe_clean_datasets["species"], "name", "species_key")

universe_clean_datasets["starships"] = add_key_column(universe_clean_datasets["starships"], "name", "starship_key")
universe_clean_datasets["starships"]["pilot_keys"] = universe_clean_datasets["starships"]["pilots"].apply(lambda value: normalize_list_keys(value, normalize_character_key))
universe_clean_datasets["starships"]["pilots"] = universe_clean_datasets["starships"]["pilots"].apply(lambda value: normalize_list_text(value, canonical_character_name))
universe_clean_datasets["starships"]["film_keys"] = universe_clean_datasets["starships"]["films"].apply(lambda value: normalize_list_keys(value, normalize_film_key))

universe_clean_datasets["vehicles"] = add_key_column(universe_clean_datasets["vehicles"], "name", "vehicle_key")
universe_clean_datasets["vehicles"]["pilot_keys"] = universe_clean_datasets["vehicles"]["pilots"].apply(lambda value: normalize_list_keys(value, normalize_character_key))
universe_clean_datasets["vehicles"]["pilots"] = universe_clean_datasets["vehicles"]["pilots"].apply(lambda value: normalize_list_text(value, canonical_character_name))
universe_clean_datasets["vehicles"]["film_keys"] = universe_clean_datasets["vehicles"]["films"].apply(lambda value: normalize_list_keys(value, normalize_film_key))

universe_clean_datasets["quotes"]["character_key"] = universe_clean_datasets["quotes"]["character_name"].apply(normalize_character_key)
universe_clean_datasets["quotes"]["character_name"] = universe_clean_datasets["quotes"]["character_name"].apply(canonical_character_name)
universe_clean_datasets["quotes"]["source_film_key"] = universe_clean_datasets["quotes"]["source"].apply(normalize_film_key)

universe_clean_datasets["weapons"] = add_key_column(universe_clean_datasets["weapons"], "name", "weapon_key")
universe_clean_datasets["weapons"]["film_keys"] = universe_clean_datasets["weapons"]["films"].apply(
    lambda value: "all_episodes" if isinstance(value, str) and value.strip().lower() == "all episodes" else normalize_list_keys(value, normalize_film_key)
)

universe_clean_datasets["droids"]["droid_key"] = universe_clean_datasets["droids"]["name"].apply(normalize_character_key)
universe_clean_datasets["droids"]["name"] = universe_clean_datasets["droids"]["name"].apply(canonical_character_name)
universe_clean_datasets["droids"]["film_keys"] = universe_clean_datasets["droids"]["films"].apply(lambda value: normalize_list_keys(value, normalize_film_key))

# Recuperamos variables comodas para seguir trabajando en el notebook.
df_characters_clean = universe_clean_datasets["characters"]
df_films_clean = universe_clean_datasets["films"]
df_planets_clean = universe_clean_datasets["planets"]
df_species_clean = universe_clean_datasets["species"]
df_starships_clean = universe_clean_datasets["starships"]
df_vehicles_clean = universe_clean_datasets["vehicles"]
df_quotes_clean = universe_clean_datasets["quotes"]
df_weapons_clean = universe_clean_datasets["weapons"]
df_droids_clean = universe_clean_datasets["droids"]

print("Datasets del universo limpiados:")
for name, df in universe_clean_datasets.items():
    print(f"{name}: {df.shape[0]} filas x {df.shape[1]} columnas")


In [ ]:
# REPORTE DE CALIDAD DE LOS DATASETS DEL UNIVERSO

universe_quality_summary = []
universe_missing_reports = {}

for name, df in universe_clean_datasets.items():
    duplicate_rows = df.duplicated().sum()
    duplicate_names = df["name"].duplicated().sum() if "name" in df.columns else np.nan
    total_cells = df.shape[0] * df.shape[1]
    total_missing = int(df.isna().sum().sum())
    missing_pct = round((total_missing / total_cells) * 100, 2) if total_cells else 0

    universe_quality_summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": duplicate_rows,
        "duplicate_names": duplicate_names,
        "total_missing": total_missing,
        "missing_pct": missing_pct,
    })

    universe_missing_reports[name] = missing_report(df)

universe_quality_summary = pd.DataFrame(universe_quality_summary)

display(universe_quality_summary)

print("Columnas con mas nulos por dataset:")
for name, report in universe_missing_reports.items():
    print("\n" + "=" * 70)
    print(name.upper())
    display(report.head(10))

In [ ]:
# TABLA RESUMEN DE ACTIVOS DEL UNIVERSO PARA POWER BI

# Esta tabla unifica activos de distinto tipo para rankings y KPIs ejecutivos.
# No sustituye a las tablas detalladas; sirve para comparar presencia general.

asset_tables = []

asset_sources = {
    "character": (df_characters_clean, "name", "character_key"),
    "planet": (df_planets_clean, "name", "planet_key"),
    "species": (df_species_clean, "name", "species_key"),
    "starship": (df_starships_clean, "name", "starship_key"),
    "vehicle": (df_vehicles_clean, "name", "vehicle_key"),
    "weapon": (df_weapons_clean, "name", "weapon_key"),
    "droid": (df_droids_clean, "name", "droid_key"),
}

for asset_type, (df, name_col, key_col) in asset_sources.items():
    temp = pd.DataFrame({
        "asset_key": df[key_col] if key_col in df.columns else df[name_col].apply(normalize_text_key),
        "asset_name": df[name_col] if name_col in df.columns else pd.Series([np.nan] * len(df)),
    })
    temp["asset_type"] = asset_type
    temp["source_dataset"] = asset_type + "s"
    temp["film_count"] = df["film_count"] if "film_count" in df.columns else np.nan
    temp["known_numeric_fields"] = df.select_dtypes(include="number").notna().sum(axis=1).values
    temp["missing_fields"] = df.isna().sum(axis=1).values
    temp["total_fields"] = df.shape[1]
    temp["data_completeness_pct"] = ((temp["total_fields"] - temp["missing_fields"]) / temp["total_fields"] * 100).round(2)
    asset_tables.append(temp)

universe_assets = pd.concat(asset_tables, ignore_index=True)

# Indicador simple de presencia interna. Mas adelante se podra combinar con afinidad de audiencia.
universe_assets["internal_presence_score"] = (
    universe_assets["film_count"].fillna(0) + universe_assets["known_numeric_fields"].fillna(0)
)

display(universe_assets.head())
print("Activos por tipo:")
display(universe_assets["asset_type"].value_counts())


In [ ]:
# EXPORTACION DE DATASETS DEL UNIVERSO LIMPIOS

for name, df in universe_clean_datasets.items():
    df.to_csv(PROCESSED_DIR / f"universe_{name}_clean.csv", index=False)

universe_quality_summary.to_csv(PROCESSED_DIR / "universe_quality_summary.csv", index=False)
universe_assets.to_csv(PROCESSED_DIR / "universe_assets.csv", index=False)

print("CSV del universo exportados en:", PROCESSED_DIR)
print("Archivos creados:")
for file in sorted(PROCESSED_DIR.glob("universe_*.csv")):
    print("-", file.name)

### Datos comerciales normalizados de peliculas

Esta tabla usa `film_key` como clave estable para cruzar datos de taquilla, encuesta y universo sin depender de diferencias de escritura en los titulos.


In [ ]:
# DATOS COMERCIALES DE PELICULAS

from io import StringIO

film_business_csv = """film_key,film_title,the_numbers_title,release_date,era,film_type,budget_usd,domestic_box_office_usd,worldwide_box_office_usd,data_status
"a_new_hope","A New Hope","Star Wars Ep. IV: A New Hope",1977-05-25,original_trilogy,saga_episode,11000000,460998007,775398007,final
"the_empire_strikes_back","The Empire Strikes Back","Star Wars Ep. V: The Empire Strikes Back",1980-05-20,original_trilogy,saga_episode,23000000,291738960,549001086,final
"return_of_the_jedi","Return of the Jedi","Star Wars Ep. VI: Return of the Jedi",1983-05-25,original_trilogy,saga_episode,32500000,316465003,482365284,final
"the_phantom_menace","The Phantom Menace","Star Wars Ep. I: The Phantom Menace",1999-05-19,prequel_trilogy,saga_episode,115000000,487574671,1046513456,final
"attack_of_the_clones","Attack of the Clones","Star Wars Ep. II: Attack of the Clones",2002-05-16,prequel_trilogy,saga_episode,115000000,310676740,656695615,final
"revenge_of_the_sith","Revenge of the Sith","Star Wars Ep. III: Revenge of the Sith",2005-05-18,prequel_trilogy,saga_episode,115000000,414378291,902891983,final
"the_force_awakens","The Force Awakens","Star Wars Ep. VII: The Force Awakens",2015-12-16,sequel_trilogy,saga_episode,533200000,936662225,2056046835,final
"rogue_one","Rogue One","Rogue One: A Star Wars Story",2016-12-14,disney_anthology,spin_off,280200000,533539991,1055083596,final
"the_last_jedi","The Last Jedi","Star Wars Ep. VIII: The Last Jedi",2017-12-13,sequel_trilogy,saga_episode,262000000,620181382,1322581071,final
"solo","Solo","Solo: A Star Wars Story",2018-05-23,disney_anthology,spin_off,330400000,213767512,393151347,final
"the_rise_of_skywalker","The Rise of Skywalker","Star Wars: The Rise of Skywalker",2019-12-18,sequel_trilogy,saga_episode,275000000,515202542,1069951814,final
"the_mandalorian_and_grogu","The Mandalorian and Grogu","Star Wars: The Mandalorian and Grogu",2026-05-20,new_theatrical_release,streaming_series_continuation,165000000,158382333,296046513,partial_current_release
"""

films_business_clean = pd.read_csv(StringIO(film_business_csv), parse_dates=["release_date"])
films_business_clean["release_year"] = films_business_clean["release_date"].dt.year
films_business_clean["profit_estimated_usd"] = films_business_clean["worldwide_box_office_usd"] - films_business_clean["budget_usd"]
films_business_clean["roi"] = (films_business_clean["profit_estimated_usd"] / films_business_clean["budget_usd"]).round(4)
films_business_clean["source_name"] = "The Numbers"
films_business_clean["source_url"] = "https://www.the-numbers.com/movies/franchise/Star-Wars"

films_business_clean = films_business_clean[[
    "film_key", "film_title", "the_numbers_title", "release_date", "release_year", "era", "film_type",
    "budget_usd", "domestic_box_office_usd", "worldwide_box_office_usd", "profit_estimated_usd", "roi",
    "data_status", "source_name", "source_url",
]]

eda_film_business_summary = films_business_clean.sort_values(
    ["worldwide_box_office_usd", "roi"], ascending=False
).reset_index(drop=True)

films_business_clean.to_csv(PROCESSED_DIR / "films_business_clean.csv", index=False)

print("Tabla comercial de peliculas:", films_business_clean.shape)
display(eda_film_business_summary)


## 7. EDA ejecutivo

Con los datos ya limpios, este bloque realiza el analisis exploratorio orientado al dashboard final.

El objetivo no es solo describir los datos, sino convertirlos en lectura de negocio:

- Que peliculas y personajes tienen mas traccion en audiencia.
- Que activos internos existen en el universo Star Wars.
- Donde hay problemas de calidad, nulos o sesgos.
- Que elementos tienen mas potencial para merchandising.

In [ ]:
# EDA 1: AUDIENCIA Y PERCEPCION

survey_kpis = pd.DataFrame([{
    "respondents": len(survey_respondents),
    "seen_any_star_wars_pct": round(survey_respondents["has_seen_any_star_wars_film_binary"].mean() * 100, 2),
    "star_wars_fan_pct": round(survey_respondents["is_star_wars_fan_binary"].mean() * 100, 2),
    "avg_movies_seen": round(survey_respondents["total_movies_seen"].mean(), 2),
    "complete_movie_ranking_pct": round(survey_respondents["has_complete_movie_ranking"].mean() * 100, 2),
    "with_demographic_info_pct": round(survey_respondents["has_demographic_info"].mean() * 100, 2),
}])

movie_views_summary = (
    survey_movies_seen
    .groupby(["episode_order", "film_key", "movie_title"], as_index=False)
    .agg(viewers=("has_seen_movie", "sum"), respondents=("respondent_id", "nunique"))
)
movie_views_summary["view_rate_pct"] = (movie_views_summary["viewers"] / movie_views_summary["respondents"] * 100).round(2)

movie_rank_summary = (
    survey_movie_rankings
    .dropna(subset=["movie_rank"])
    .groupby(["episode_order", "film_key", "movie_title"], as_index=False)
    .agg(
        avg_rank=("movie_rank", "mean"),
        median_rank=("movie_rank", "median"),
        ranking_responses=("respondent_id", "nunique"),
        first_place_votes=("movie_rank", lambda s: (s == 1).sum()),
    )
)
movie_rank_summary["avg_rank"] = movie_rank_summary["avg_rank"].round(2)
movie_rank_summary["first_place_pct"] = (movie_rank_summary["first_place_votes"] / movie_rank_summary["ranking_responses"] * 100).round(2)
movie_rank_summary["preference_score"] = (7 - movie_rank_summary["avg_rank"]).round(2)

character_opinion_summary = (
    survey_character_opinions
    .groupby(["character_key", "character_name"], as_index=False)
    .agg(
        avg_opinion_score=("opinion_score", "mean"),
        opinion_responses=("opinion_label", lambda s: s.notna().sum()),
        favorable_responses=("opinion_score", lambda s: (s > 0).sum()),
        unfavorable_responses=("opinion_score", lambda s: (s < 0).sum()),
        unfamiliar_responses=("is_unfamiliar", "sum"),
        total_rows=("respondent_id", "count"),
    )
)
character_opinion_summary["avg_opinion_score"] = character_opinion_summary["avg_opinion_score"].round(3)
character_opinion_summary["favorable_pct"] = (character_opinion_summary["favorable_responses"] / character_opinion_summary["opinion_responses"] * 100).round(2)
character_opinion_summary["unfavorable_pct"] = (character_opinion_summary["unfavorable_responses"] / character_opinion_summary["opinion_responses"] * 100).round(2)
character_opinion_summary["unfamiliar_pct"] = (character_opinion_summary["unfamiliar_responses"] / character_opinion_summary["opinion_responses"] * 100).round(2)
character_opinion_summary = character_opinion_summary.sort_values(["avg_opinion_score", "favorable_pct"], ascending=False)

fan_by_age = (
    survey_respondents
    .dropna(subset=["age"])
    .groupby("age", as_index=False)
    .agg(
        respondents=("respondent_id", "nunique"),
        fan_rate_pct=("is_star_wars_fan_binary", lambda s: round(s.mean() * 100, 2)),
        avg_movies_seen=("total_movies_seen", "mean"),
    )
)
fan_by_age["avg_movies_seen"] = fan_by_age["avg_movies_seen"].round(2)

fan_by_gender = (
    survey_respondents
    .dropna(subset=["gender"])
    .groupby("gender", as_index=False)
    .agg(
        respondents=("respondent_id", "nunique"),
        fan_rate_pct=("is_star_wars_fan_binary", lambda s: round(s.mean() * 100, 2)),
        avg_movies_seen=("total_movies_seen", "mean"),
    )
)
fan_by_gender["avg_movies_seen"] = fan_by_gender["avg_movies_seen"].round(2)

print("KPIs encuesta")
display(survey_kpis)

print("Peliculas mas vistas")
display(movie_views_summary.sort_values("view_rate_pct", ascending=False))

print("Ranking medio de peliculas: menor avg_rank = mejor preferencia")
display(movie_rank_summary.sort_values("avg_rank"))

print("Personajes mejor valorados")
display(character_opinion_summary.head(10))

print("Fan rate por edad")
display(fan_by_age)

print("Fan rate por genero")
display(fan_by_gender)


In [ ]:
# EDA 2: CONTENIDO INTERNO DEL UNIVERSO STAR WARS

universe_overview = pd.DataFrame([
    {"asset_type": "characters", "count": len(df_characters_clean)},
    {"asset_type": "films", "count": len(df_films_clean)},
    {"asset_type": "planets", "count": len(df_planets_clean)},
    {"asset_type": "species", "count": len(df_species_clean)},
    {"asset_type": "starships", "count": len(df_starships_clean)},
    {"asset_type": "vehicles", "count": len(df_vehicles_clean)},
    {"asset_type": "weapons", "count": len(df_weapons_clean)},
    {"asset_type": "droids", "count": len(df_droids_clean)},
    {"asset_type": "quotes", "count": len(df_quotes_clean)},
])

character_species_summary = (
    df_characters_clean
    .groupby("species", dropna=False, as_index=False)
    .agg(characters=("id", "count"))
    .sort_values("characters", ascending=False)
)

character_gender_summary = (
    df_characters_clean
    .groupby("gender", dropna=False, as_index=False)
    .agg(characters=("id", "count"))
    .sort_values("characters", ascending=False)
)

character_homeworld_summary = (
    df_characters_clean
    .groupby("homeworld", dropna=False, as_index=False)
    .agg(characters=("id", "count"))
    .sort_values("characters", ascending=False)
)

planet_business_summary = df_planets_clean[[
    "planet_key", "name", "population", "diameter", "climate", "terrain", "resident_keys", "resident_count", "film_keys", "film_count"
]].copy().sort_values(["film_count", "resident_count", "population"], ascending=False)

starship_business_summary = df_starships_clean[[
    "starship_key", "name", "starship_class", "manufacturer", "cost_in_credits", "length", "crew",
    "passengers", "cargo_capacity", "pilot_keys", "pilot_count", "film_keys", "film_count"
]].copy().sort_values(["film_count", "pilot_count", "cost_in_credits"], ascending=False)

weapon_business_summary = df_weapons_clean[[
    "weapon_key", "name", "type", "manufacturer", "cost_in_credits", "length", "film_keys", "film_count"
]].copy().sort_values(["film_count", "cost_in_credits"], ascending=False)

quote_character_summary = (
    df_quotes_clean
    .groupby(["character_key", "character_name"], as_index=False)
    .agg(quote_count=("quote", "count"))
    .sort_values("quote_count", ascending=False)
)

print("Resumen de activos internos")
display(universe_overview)

print("Top especies por numero de personajes")
display(character_species_summary.head(10))

print("Distribucion de genero de personajes")
display(character_gender_summary)

print("Top planetas por presencia narrativa")
display(planet_business_summary.head(10))

print("Top naves por presencia/capacidad comercial")
display(starship_business_summary.head(10))

print("Top armas por presencia")
display(weapon_business_summary.head(10))

print("Personajes con mas citas registradas")
display(quote_character_summary.head(10))


In [ ]:
# EDA 3: SESGOS, GOBERNANZA Y CALIDAD DEL DATO

survey_missing_top = missing_report(df_survey_clean).reset_index().rename(columns={"index": "column"})
survey_missing_top["dataset"] = "survey"

universe_missing_long = []
for dataset_name, report in universe_missing_reports.items():
    temp = report.reset_index().rename(columns={"index": "column"})
    temp["dataset"] = dataset_name
    universe_missing_long.append(temp)

universe_missing_long = pd.concat(universe_missing_long, ignore_index=True) if universe_missing_long else pd.DataFrame()

governance_missing_top = pd.concat(
    [survey_missing_top[["dataset", "column", "nulos", "porcentaje"]], universe_missing_long[["dataset", "column", "nulos", "porcentaje"]]],
    ignore_index=True,
).sort_values(["porcentaje", "nulos"], ascending=False)

survey_sample_bias = pd.DataFrame([
    {
        "risk": "Muestra orientada a personas que conocen Star Wars",
        "evidence": f"{survey_kpis.loc[0, 'seen_any_star_wars_pct']}% declara haber visto alguna pelicula.",
        "business_impact": "Puede sobreestimar la demanda real del publico general.",
    },
    {
        "risk": "Sesgo fan",
        "evidence": f"{survey_kpis.loc[0, 'star_wars_fan_pct']}% se declara fan entre respuestas validas.",
        "business_impact": "Las preferencias pueden favorecer personajes iconicos y no nichos de crecimiento.",
    },
    {
        "risk": "Datos demograficos incompletos",
        "evidence": f"{survey_kpis.loc[0, 'with_demographic_info_pct']}% tiene alguna informacion demografica util.",
        "business_impact": "La segmentacion por edad, genero, ingresos o region debe interpretarse con cautela.",
    },
    {
        "risk": "Datos internos incompletos",
        "evidence": "Planetas, naves, armas y droides tienen campos economicos o fisicos con nulos.",
        "business_impact": "El potencial comercial no debe calcularse solo con coste, tamano o capacidad.",
    },
])

print("Top problemas de nulos para pagina de gobernanza")
display(governance_missing_top.head(20))

print("Riesgos de sesgo detectados")
display(survey_sample_bias)

In [ ]:
# EDA 4: INDICE DE POTENCIAL DE MERCHANDISING

character_opportunities = character_opinion_summary.copy()

characters_for_merge = df_characters_clean[[
    "character_key", "name", "species", "gender", "homeworld", "height", "weight"
]].rename(columns={"name": "universe_character_name"})

quote_for_merge = quote_character_summary[["character_key", "quote_count"]].copy()

asset_quality_for_merge = universe_assets[universe_assets["asset_type"] == "character"][[
    "asset_key", "data_completeness_pct", "internal_presence_score"
]].copy().rename(columns={"asset_key": "character_key"})

character_opportunities = character_opportunities.merge(characters_for_merge, on="character_key", how="left")
character_opportunities = character_opportunities.merge(quote_for_merge, on="character_key", how="left")
character_opportunities = character_opportunities.merge(asset_quality_for_merge, on="character_key", how="left")

character_opportunities["quote_count"] = character_opportunities["quote_count"].fillna(0)
character_opportunities["is_in_universe_dataset"] = character_opportunities["universe_character_name"].notna().astype(int)
character_opportunities["data_completeness_pct"] = character_opportunities["data_completeness_pct"].fillna(0)
character_opportunities["internal_presence_score"] = character_opportunities["internal_presence_score"].fillna(0) + character_opportunities["quote_count"]

# Escalado de componentes a 0-100.
character_opportunities["audience_affinity_score"] = (((character_opportunities["avg_opinion_score"] + 2) / 4) * 100).round(2)
character_opportunities["familiarity_score"] = (100 - character_opportunities["unfamiliar_pct"]).round(2)
max_presence = character_opportunities["internal_presence_score"].max()
character_opportunities["internal_presence_score_scaled"] = np.where(
    max_presence > 0,
    (character_opportunities["internal_presence_score"] / max_presence * 100).round(2),
    0,
)
character_opportunities["data_quality_score"] = character_opportunities["data_completeness_pct"].round(2)

character_opportunities["merchandising_potential_index"] = (
    character_opportunities["audience_affinity_score"] * 0.45
    + character_opportunities["familiarity_score"] * 0.25
    + character_opportunities["internal_presence_score_scaled"] * 0.20
    + character_opportunities["data_quality_score"] * 0.10
).round(2)

presence_median = character_opportunities["internal_presence_score_scaled"].median()
audience_median = character_opportunities["audience_affinity_score"].median()


def opportunity_quadrant(row):
    high_presence = row["internal_presence_score_scaled"] >= presence_median
    high_audience = row["audience_affinity_score"] >= audience_median
    if high_presence and high_audience:
        return "priority_campaign"
    if high_presence and not high_audience:
        return "repositioning_needed"
    if not high_presence and high_audience:
        return "hidden_opportunity"
    return "low_priority"


character_opportunities["opportunity_quadrant"] = character_opportunities.apply(opportunity_quadrant, axis=1)
character_opportunities = character_opportunities.sort_values("merchandising_potential_index", ascending=False)

movie_opportunities = movie_views_summary.merge(
    movie_rank_summary[["film_key", "avg_rank", "preference_score", "first_place_pct"]],
    on="film_key",
    how="left",
)
movie_opportunities["movie_campaign_score"] = (
    movie_opportunities["view_rate_pct"] * 0.45
    + (movie_opportunities["preference_score"] / 6 * 100) * 0.45
    + movie_opportunities["first_place_pct"] * 0.10
).round(2)
movie_opportunities = movie_opportunities.sort_values("movie_campaign_score", ascending=False)


movie_audience_summary = movie_views_summary.merge(
    movie_rank_summary[["film_key", "avg_rank", "preference_score", "ranking_responses", "first_place_pct"]],
    on="film_key",
    how="left",
)

eda_movie_commercial_audience_summary = films_business_clean.merge(
    movie_audience_summary,
    on="film_key",
    how="left",
)
eda_movie_commercial_audience_summary["movie_title"] = eda_movie_commercial_audience_summary["film_key"].map(FILM_DISPLAY_NAMES)
eda_movie_commercial_audience_summary["is_in_survey"] = eda_movie_commercial_audience_summary["viewers"].notna().astype(int)

print("Top personajes por indice de merchandising")
display(character_opportunities[[
    "character_key", "character_name", "merchandising_potential_index", "opportunity_quadrant",
    "audience_affinity_score", "familiarity_score", "internal_presence_score_scaled",
    "data_quality_score", "avg_opinion_score", "quote_count", "species", "gender", "homeworld"
]].head(12))

print("Cuadrante de oportunidades")
display(character_opportunities["opportunity_quadrant"].value_counts())

print("Oportunidad por pelicula")
display(movie_opportunities)

print("Resumen comercial + audiencia por pelicula")
display(eda_movie_commercial_audience_summary)


In [ ]:
# CONCLUSIONES AUTOMATICAS DEL EDA

best_movie_by_views = movie_views_summary.sort_values("view_rate_pct", ascending=False).iloc[0]
best_movie_by_rank = movie_rank_summary.sort_values("avg_rank").iloc[0]
best_character = character_opportunities.iloc[0]
best_business_movie = eda_film_business_summary.iloc[0]
best_roi_movie = films_business_clean.sort_values("roi", ascending=False).iloc[0]
most_common_species = character_species_summary.iloc[0]
most_common_gender = character_gender_summary.iloc[0]

eda_conclusions = pd.DataFrame([
    {
        "area": "Audiencia",
        "finding": f"La pelicula mas vista es {best_movie_by_views['movie_title']} con un {best_movie_by_views['view_rate_pct']}% de visionado.",
        "business_reading": "Es una candidata fuerte para campanas de alto reconocimiento.",
    },
    {
        "area": "Preferencia",
        "finding": f"La pelicula con mejor ranking medio es {best_movie_by_rank['movie_title']} con avg_rank {best_movie_by_rank['avg_rank']}.",
        "business_reading": "La preferencia declarada no siempre coincide con la simple exposicion.",
    },
    {
        "area": "Personajes",
        "finding": f"El personaje con mayor indice de merchandising es {best_character['character_name']}.",
        "business_reading": "Conviene priorizar personajes con alta afinidad, familiaridad y presencia narrativa.",
    },
    {
        "area": "Representacion interna",
        "finding": f"La especie mas frecuente en characters es {most_common_species['species']} con {most_common_species['characters']} personajes.",
        "business_reading": "El universo esta concentrado en ciertos grupos, lo que puede limitar diversidad de campanas.",
    },
    {
        "area": "Negocio",
        "finding": f"La pelicula con mayor taquilla mundial es {best_business_movie['film_title']} con ${best_business_movie['worldwide_box_office_usd']:,.0f}.",
        "business_reading": "El rendimiento comercial complementa la lectura de audiencia y ayuda a priorizar oportunidades de franquicia.",
    },
    {
        "area": "Rentabilidad",
        "finding": f"La pelicula con mayor ROI estimado es {best_roi_movie['film_title']} con ROI {best_roi_movie['roi']:.2f}.",
        "business_reading": "Las peliculas clasicas pueden destacar proporcionalmente aunque las nuevas tengan mayor taquilla absoluta.",
    },
    {
        "area": "Gobernanza",
        "finding": "Existen nulos relevantes en campos fisicos, economicos y demograficos.",
        "business_reading": "Las decisiones deben combinar datos cuantitativos con criterio editorial y de negocio.",
    },
])

display(eda_conclusions)


In [ ]:
# EXPORTACION DE TABLAS RESUMEN DEL EDA

eda_outputs = {
    "eda_survey_kpis.csv": survey_kpis,
    "eda_movie_views_summary.csv": movie_views_summary,
    "eda_movie_rank_summary.csv": movie_rank_summary,
    "eda_character_opinion_summary.csv": character_opinion_summary,
    "eda_fan_by_age.csv": fan_by_age,
    "eda_fan_by_gender.csv": fan_by_gender,
    "eda_universe_overview.csv": universe_overview,
    "eda_character_species_summary.csv": character_species_summary,
    "eda_character_gender_summary.csv": character_gender_summary,
    "eda_character_homeworld_summary.csv": character_homeworld_summary,
    "eda_planet_business_summary.csv": planet_business_summary,
    "eda_starship_business_summary.csv": starship_business_summary,
    "eda_weapon_business_summary.csv": weapon_business_summary,
    "eda_quote_character_summary.csv": quote_character_summary,
    "eda_film_business_summary.csv": eda_film_business_summary,
    "eda_movie_commercial_audience_summary.csv": eda_movie_commercial_audience_summary,
    "eda_governance_missing_top.csv": governance_missing_top,
    "eda_survey_sample_bias.csv": survey_sample_bias,
    "eda_character_merchandising_opportunities.csv": character_opportunities,
    "eda_movie_opportunities.csv": movie_opportunities,
    "eda_conclusions.csv": eda_conclusions,
}

for filename, df in eda_outputs.items():
    df.to_csv(PROCESSED_DIR / filename, index=False)

print("Tablas de EDA exportadas:")
for filename in sorted(eda_outputs):
    print("-", filename)


### Validacion de claves y relaciones

Checks para confirmar que las claves usadas en Power BI no generan personajes duplicados por escritura, peliculas huerfanas o joins incompletos entre encuesta, universo y tablas comerciales.


In [ ]:
# VALIDACION DE CLAVES Y RELACIONES

quality_checks = []


def add_quality_check(check_name, status, affected_rows, detail):
    quality_checks.append({
        "check_name": check_name,
        "status": status,
        "affected_rows": int(affected_rows),
        "detail": detail,
    })


survey_character_keys = set(survey_character_opinions["character_key"].dropna().unique())
universe_character_keys = set(df_characters_clean["character_key"].dropna().unique())
missing_survey_characters = sorted(survey_character_keys - universe_character_keys)
add_quality_check(
    "survey_characters_exist_in_universe",
    "pass" if not missing_survey_characters else "fail",
    len(missing_survey_characters),
    ", ".join(missing_survey_characters) if missing_survey_characters else "All survey character keys match universe_characters_clean.",
)

missing_merchandising_matches = character_opportunities.loc[
    character_opportunities["is_in_universe_dataset"].eq(0), "character_key"
].dropna().tolist()
add_quality_check(
    "merchandising_characters_have_universe_match",
    "pass" if not missing_merchandising_matches else "fail",
    len(missing_merchandising_matches),
    ", ".join(missing_merchandising_matches) if missing_merchandising_matches else "All merchandising characters have a universe match.",
)

survey_film_keys = set(survey_movies_seen["film_key"].dropna().unique()) | set(survey_movie_rankings["film_key"].dropna().unique())
universe_film_keys = set(df_films_clean["film_key"].dropna().unique())
business_film_keys = set(films_business_clean["film_key"].dropna().unique())
missing_survey_films_in_universe = sorted(survey_film_keys - universe_film_keys)
missing_survey_films_in_business = sorted(survey_film_keys - business_film_keys)
add_quality_check(
    "survey_films_exist_in_universe_films",
    "pass" if not missing_survey_films_in_universe else "fail",
    len(missing_survey_films_in_universe),
    ", ".join(missing_survey_films_in_universe) if missing_survey_films_in_universe else "All survey film keys match universe_films_clean.",
)
add_quality_check(
    "survey_films_exist_in_business_films",
    "pass" if not missing_survey_films_in_business else "fail",
    len(missing_survey_films_in_business),
    ", ".join(missing_survey_films_in_business) if missing_survey_films_in_business else "All survey film keys match films_business_clean.",
)

for dataset_name, df, key_col in [
    ("universe_characters_clean", df_characters_clean, "character_key"),
    ("universe_films_clean", df_films_clean, "film_key"),
    ("films_business_clean", films_business_clean, "film_key"),
]:
    duplicated_keys = df.loc[df[key_col].notna() & df[key_col].duplicated(keep=False), key_col].unique().tolist()
    add_quality_check(
        f"{dataset_name}_{key_col}_unique",
        "pass" if not duplicated_keys else "fail",
        len(duplicated_keys),
        ", ".join(map(str, duplicated_keys)) if duplicated_keys else f"{key_col} is unique in {dataset_name}.",
    )


def collect_list_keys(df, column):
    if column not in df.columns:
        return set()
    keys = set()
    for value in df[column].dropna():
        keys.update([item.strip() for item in str(value).split(",") if item.strip()])
    return keys

list_relationship_checks = [
    ("planet_resident_keys_in_characters", df_planets_clean, "resident_keys", universe_character_keys),
    ("starship_pilot_keys_in_characters", df_starships_clean, "pilot_keys", universe_character_keys),
    ("vehicle_pilot_keys_in_characters", df_vehicles_clean, "pilot_keys", universe_character_keys),
    ("planet_film_keys_in_films", df_planets_clean, "film_keys", universe_film_keys | {"all_episodes"}),
    ("starship_film_keys_in_films", df_starships_clean, "film_keys", universe_film_keys | {"all_episodes"}),
    ("vehicle_film_keys_in_films", df_vehicles_clean, "film_keys", universe_film_keys | {"all_episodes"}),
    ("weapon_film_keys_in_films", df_weapons_clean, "film_keys", universe_film_keys | {"all_episodes"}),
    ("droid_film_keys_in_films", df_droids_clean, "film_keys", universe_film_keys | {"all_episodes"}),
]

for check_name, df, column, allowed_keys in list_relationship_checks:
    used_keys = collect_list_keys(df, column)
    missing_keys = sorted(used_keys - allowed_keys)
    add_quality_check(
        check_name,
        "pass" if not missing_keys else "warn",
        len(missing_keys),
        ", ".join(missing_keys[:30]) if missing_keys else f"All keys from {column} have a matching dimension key.",
    )


relationship_quality_checks = pd.DataFrame(quality_checks)
relationship_quality_checks.to_csv(PROCESSED_DIR / "eda_relationship_quality_checks.csv", index=False)

display(relationship_quality_checks)

if (relationship_quality_checks["status"] == "fail").any():
    raise ValueError("Hay checks de relacion fallidos. Revisar relationship_quality_checks.")
